# 01. 학습 파이프라인 — 모든 모델의 학습·추론 실행

이 노트북은 STEP 2~7의 학습·추론을 순서대로 실행합니다. 결과 표·그림·해석은 `02_AI예측모델개발.ipynb`에서 봅니다.

- 학습 코드는 `scripts/*.py`(단계별 스크립트)와 `power_utils/`(공통 함수)에 있고, 이 노트북은 각 셀에서 스크립트를 순서대로 실행합니다. 셀마다 별도 프로세스로 돌리고 로그를 `outputs/logs/`에 남깁니다.
- **`RETRAIN = False`(기본)**: 결과 파일이 이미 있으면 그 단계를 건너뛰고 저장된 결과를 씁니다(몇 초).
- **`RETRAIN = True`**: 처음부터 다시 학습합니다. 셀마다 이번 실행과 같은 seed·스레드 수(트리·통계·딥러닝 1개, Chronos·STEP 6·7 2개)를 넘겨, 같은 환경에서는 지표가 소수점까지 똑같이 나옵니다(02 노트북 [표 M-39]). 딥러닝은 스레드 수가 달라지면 결과가 조금 달라질 수 있습니다. CPU에서는 오래 걸립니다.

| 단계 | 내용 | 이번 실행 시간(CPU, 병렬 실행 기준) |
|---|---|---|
| STEP 2 | 가이드북 RNN 18회 · RF | 31분 · 30초 |
| STEP 3 | LightGBM·XGBoost·CatBoost 튜닝·학습(작업 18개), MSTL·AutoARIMA | 33분, 68분 |
| STEP 4 | NHITS·LSTM·TFT 학습 198회 | 7시간 9분 |
| STEP 5 | Chronos-2 zero-shot·공변량, Chronos-2-small LoRA | 약 1시간, 3시간 40분, 3시간 30분 |
| STEP 6 | 앙상블·컨포멀·확률보정 | 1분 |
| STEP 7 | 선정 → 테스트 1회 예측 → SHAP·민감도·재현성 | 1시간 10분 |
| 추가 실험 | Chronos-2 임베딩 + 피크 분류기(테스트 공개 후, 결과는 노트북 03) | 13분 |

> 이 노트북은 같은 작업을 순서대로 하나씩 돌리므로, `RETRAIN = True`면 위 시간보다 더 걸립니다(CPU 기준 하루 이상). 테스트 구간(9/1~9/14)은 STEP 7의 `predict_test.py`에서만, 선정 규칙을 정한 뒤 한 번 씁니다.

실행 설정입니다. 프로젝트 폴더를 찾아 공통 함수를 불러오고, 다시 학습할지(`RETRAIN`)를 정합니다. `need()`는 '결과 파일이 이미 있으면 건너뛸지'를 판단하는 작은 도우미입니다. 출력된 `RETRAIN` 값으로 지금 어떤 방식으로 실행되는지 확인합니다.

In [1]:
# [코드 P-01] 실행 설정: 프로젝트 경로, 다시 학습할지 여부, 건너뛰기 판단
import sys, json, time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'power_utils').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import power_utils as pu

RETRAIN = False          # True면 처음부터 다시 학습(CPU 기준 하루 이상), False면 저장된 결과 사용
T = pu.OUT / 'tables'


def need(*paths):
    """다시 실행해야 하면 True. RETRAIN=False이고 결과 파일이 모두 있으면 건너뛴다."""
    missing = [p for p in paths if not Path(p).exists()]
    if not RETRAIN and not missing:
        print('저장된 결과 사용:', ', '.join(f'{Path(p).name}({time.strftime("%m-%d %H:%M", time.localtime(Path(p).stat().st_mtime))})' for p in paths))
        return False
    return True


t_start = time.time()
print(f'프로젝트: {ROOT.name} | RETRAIN = {RETRAIN} | 출력 폴더: {pu.OUT.name}')

프로젝트: hourly_max_pipeline | RETRAIN = False | 출력 폴더: outputs


In [2]:
# Parameters
RETRAIN = False


**STEP 2 — 가이드북 베이스라인 재평가.** 가이드북 RNN을 같은 타깃(시간 피크)·같은 시간 fold로 fold마다 seed 3개씩 학습하고(작은 RNN은 스레드 1개가 가장 빨라 fold별로 실행), 가이드북 RF를 무작위 분할과 시간 fold로 평가합니다. 끝나면 `guidebook_rnn_epochs.csv`(best epoch)와 `guidebook_rf_split_scores.csv`가 생깁니다.

In [3]:
# [코드 P-02] STEP 2: 가이드북 RNN(6 fold × seed 3)과 RF(무작위 70:30 vs 시간 fold)
if need(T / 'guidebook_rnn_epochs.csv'):
    for f in ['F1', 'F2', 'F3', 'F4', 'F5', 'F6']:
        pu.run_script('train_guidebook_rnn.py', ['--folds', f, '--no-aggregate'], f'guidebook_rnn_{f}.log', threads=1)
    pu.run_script('train_guidebook_rnn.py', ['--resume'], 'guidebook_rnn.log')
if need(T / 'guidebook_rf_split_scores.csv'):
    pu.run_script('train_guidebook_rf.py', [], 'guidebook_rf.log')

저장된 결과 사용: guidebook_rnn_epochs.csv(10-04 20:59)
저장된 결과 사용: guidebook_rf_split_scores.csv(10-04 21:01)


**STEP 3 — 통계 모델과 트리 앙상블.** LightGBM·XGBoost·CatBoost는 (모델, 트랙, 시나리오) 18개 작업마다 Optuna로 하이퍼파라미터와 최근성 가중치를 고른 뒤 seed 5개 × 6 fold로 학습하고, MSTL·AutoARIMA는 fold마다 한 번 학습해 롤링 예측합니다. 끝나면 `tree_best_params.csv`와 통계 모델 예측 파일이 생깁니다.

In [4]:
# [코드 P-03] STEP 3: 트리 18개 작업(튜닝 + seed 5 × 6 fold), MSTL·AutoARIMA
if need(T / 'tree_best_params.csv', T / 'feature_importance_gain.csv'):
    pu.run_script('train_trees.py', [], 'trees_all.log', threads=1)
if need(pu.OUT / 'preds' / 'oof_mstl_A_S0.parquet', pu.OUT / 'preds' / 'oof_arima_B_S2.parquet'):
    pu.run_script('train_stats.py', [], 'stats.log', threads=1)

저장된 결과 사용: tree_best_params.csv(10-04 21:29), feature_importance_gain.csv(10-04 21:29)
저장된 결과 사용: oof_mstl_A_S0.parquet(10-04 21:51), oof_arima_B_S2.parquet(10-04 21:51)


**STEP 4 — 딥러닝(NHITS·LSTM·TFT).** 학습 단위 목록(`outputs/logs/neural_units.txt`, 모델:트랙:시나리오@fold)을 하나씩 학습한 뒤 결과를 모읍니다. TFT는 CPU에서 매우 느려 최근 3 fold만 씁니다. 끝나면 `neural_runs.csv`(fold·seed별 best step·학습 시간)가 생깁니다.

In [5]:
# [코드 P-04] STEP 4: 딥러닝 학습 단위(198회)를 하나씩 실행한 뒤 모으기
if need(T / 'neural_runs.csv'):
    units = [u.split() for u in (pu.OUT / 'logs' / 'neural_units.txt').read_text(encoding='utf-8').splitlines() if u.strip()]
    for job, mode in units:
        log = 'neural/' + job.replace(':', '_').replace('@', '_') + '.log'
        pu.run_script('train_neural.py', ['--mode', mode, '--jobs', job, '--no-aggregate'], log, threads=1, tail=1)
    jobs = sorted({job.split('@')[0] for job, _ in units})
    pu.run_script('train_neural.py', ['--aggregate', '--jobs', *jobs], 'neural_aggregate.log')

저장된 결과 사용: neural_runs.csv(10-05 05:53)


**STEP 5 — 시계열 파운데이션 모델(Chronos-2).** 작업 목록(`outputs/logs/chronos_units.txt`)대로 단변량 zero-shot(문맥 길이 5가지), 공변량 zero-shot(S0·S1·S2), Chronos-2-small LoRA 미세조정을 실행하고 모읍니다. 처음 실행할 때 Hugging Face에서 가중치(amazon/chronos-2 478MB, autogluon/chronos-2-small 112MB)를 받습니다.

> **실행 방식 확인**: 이 환경에는 LoRA에 필요한 `peft` 패키지가 설치되어 있지 않습니다(`chronos.chronos2.pipeline.is_peft_available()` = False). 이 경우 chronos-forecasting 2.3.2의 `fit(finetune_mode='lora')`는 경고를 낸 뒤 **전체 미세조정**으로 실행합니다. 따라서 이 노트북의 'LoRA' 결과는 chronos-2-small(2천8백만 파라미터) 전체를 미세조정한 결과입니다(이름은 실행 당시 설정을 그대로 둠).

In [6]:
# [코드 P-05] STEP 5: Chronos-2 zero-shot·공변량·LoRA(작업 111개) 실행 후 모으기
if need(T / 'chronos_all_scores.csv', T / 'chronos_lora_runs.csv'):
    units = (pu.OUT / 'logs' / 'chronos_units.txt').read_text(encoding='utf-8').split()
    pu.run_script('run_chronos.py', ['--jobs', *units], 'chronos_all.log', threads=2)

저장된 결과 사용: chronos_all_scores.csv(10-05 07:29), chronos_lora_runs.csv(10-05 07:29)


**STEP 6 — 앙상블·불확실성·확률보정.** 앞 단계의 검증 예측만으로 앙상블 가중치, 컨포멀 구간 보정, 피크 위험 확률(분위수 CDF·분류기·스태킹)과 isotonic 보정, 경보 임계값을 '앞 fold로 맞추고 다음 fold에 적용'하는 방식으로 만듭니다. 테스트에 쓸 설정은 `outputs/models/step6_final_config.pkl`에 저장됩니다.

In [7]:
# [코드 P-06] STEP 6: 앙상블·컨포멀·확률보정·임계값(검증 예측만 사용)
if need(T / 'step6_final_config.json', pu.OUT / 'models' / 'step6_final_config.pkl'):
    pu.run_script('step6_ensemble.py', [], 'step6.log', threads=2)

저장된 결과 사용: step6_final_config.json(10-05 05:58), step6_final_config.pkl(10-05 17:17)


**STEP 7 — 최종 선정과 테스트 1회 평가.** 먼저 테스트 전에 정한 선정 규칙을 검증 결과에만 적용해 트랙별 모델을 고르고(`selection_result.json`), 그다음 테스트 구간을 단 한 번 예측·채점합니다(`test_scores.csv`). 마지막으로 SHAP 영향요인, 복사일 제외 민감도, 같은 seed 재실행 재현성 확인을 합니다.

In [8]:
# [코드 P-07] STEP 7: 선정(테스트 미사용) → 테스트 1회 예측 → SHAP·민감도·재현성
if need(T / 'selection_result.json'):
    pu.run_script('step7_select.py', [], 'step7_select.log')
sel = json.loads((T / 'selection_result.json').read_text(encoding='utf-8'))
print(f"선정: 트랙 A {sel['A']} + {sel['A_alarm']} | 트랙 B {sel['B']} + {sel['B_alarm']} (선정 시각 {sel['선정 시각']})")
if need(T / 'test_scores.csv'):
    pu.run_script('predict_test.py', ['--models', 'all'], 'predict_test.log', threads=2)
if need(T / 'shap_importance_A.csv', T / 'sensitivity_copied_days.csv', T / 'reproducibility_diff.csv'):
    pu.run_script('step7_extras.py', ['--select', f"A={sel['A']}", f"B={sel['B']}"], 'step7_extras.log', threads=2)

저장된 결과 사용: selection_result.json(10-05 05:59)
선정: 트랙 A chronos2_zs:S0 + 스태킹 | 트랙 B chronos2_cov:S1 + 경로2 분류기(가중) (선정 시각 2026-10-05 05:59:16)
저장된 결과 사용: test_scores.csv(10-05 06:54)
저장된 결과 사용: shap_importance_A.csv(10-05 06:57), sensitivity_copied_days.csv(10-05 06:04), reproducibility_diff.csv(10-05 06:10)


실행 결과를 한눈에 확인합니다. 단계별 주요 결과 파일이 모두 있는지, 언제 만들어졌는지 보여 주며, 모두 '있음'이면 `02_AI예측모델개발.ipynb`를 실행해 표·그림·해석을 볼 수 있습니다.

In [9]:
# [코드 P-08] 단계별 결과 파일 점검
import pandas as pd
files = {'STEP 2': [T / 'guidebook_rnn_epochs.csv', T / 'guidebook_rf_split_scores.csv'],
         'STEP 3': [T / 'tree_best_params.csv', pu.OUT / 'preds' / 'oof_mstl_A_S0.parquet'],
         'STEP 4': [T / 'neural_runs.csv'], 'STEP 5': [T / 'chronos_all_scores.csv', T / 'chronos_lora_runs.csv'],
         'STEP 6': [T / 'step6_final_config.json'],
         'STEP 7': [T / 'selection_result.json', T / 'test_scores.csv', T / 'reproducibility_diff.csv']}
rows = [{'단계': s, '파일': p.name, '있음': p.exists(),
         '수정 시각': time.strftime('%Y-%m-%d %H:%M', time.localtime(p.stat().st_mtime)) if p.exists() else ''}
        for s, ps in files.items() for p in ps]
display(pd.DataFrame(rows))
cv = pd.read_csv(pu.CV_PATH, encoding='utf-8-sig')
print(f"cv_scores.csv: {len(cv):,}행, 모델 {cv['model'].nunique()}종 | 이 노트북 실행 시간 {(time.time() - t_start) / 60:.1f}분")
assert all(r['있음'] for r in rows), '빠진 결과 파일이 있습니다'

,단계,파일,있음,수정 시각
0,STEP 2,guidebook_rnn_epochs.csv,True,2026-10-04 20:59
1,STEP 2,guidebook_rf_split_scores.csv,True,2026-10-04 21:01
2,STEP 3,tree_best_params.csv,True,2026-10-04 21:29
3,STEP 3,oof_mstl_A_S0.parquet,True,2026-10-04 21:51
4,STEP 4,neural_runs.csv,True,2026-10-05 05:53
5,STEP 5,chronos_all_scores.csv,True,2026-10-05 07:29
6,STEP 5,chronos_lora_runs.csv,True,2026-10-05 07:29
7,STEP 6,step6_final_config.json,True,2026-10-05 05:58
8,STEP 7,selection_result.json,True,2026-10-05 05:59
9,STEP 7,test_scores.csv,True,2026-10-05 06:54


cv_scores.csv: 4,065행, 모델 24종 | 이 노트북 실행 시간 0.0분


**추가 실험(테스트 공개 후) — 파운데이션 모델 임베딩 기반 피크 위험 확률.** Chronos-2가 과거 전력값을 요약한 벡터(임베딩)를 STEP 6 피크 분류기 입력에 더해, 같은 조건에서 피크를 더 잘 맞히는지 봅니다. 최종 모델 선정이 끝난 뒤의 실험이라 비교는 검증 fold로만 하고, 선정은 바꾸지 않습니다. 결과는 `03_AI_확장실험.ipynb`에서 봅니다.

In [10]:
# [코드 P-09] 추가 실험: Chronos-2 임베딩 + 피크 분류기(검증 F2~F6 비교, 테스트는 참고)
if need(T / 'X01_embed_peak_scores.csv', T / 'X07_embed_info.csv'):
    pu.run_script('ext_embed_peak.py', [], 'ext_embed_peak.log', threads=2)
x = pd.read_csv(T / 'X01_embed_peak_scores.csv', encoding='utf-8-sig')
print(f"추가 실험 결과: {len(x)}행(트랙 × 변형 × 경로) → 03_AI_확장실험.ipynb")

저장된 결과 사용: X01_embed_peak_scores.csv(10-05 14:09), X07_embed_info.csv(10-05 14:09)
추가 실험 결과: 12행(트랙 × 변형 × 경로) → 03_AI_확장실험.ipynb
